## Data Gap Handling (Upsampling and Interpolation)

In [7]:
import pandas as pd

from load_data import get_production_data

In [8]:
prod_df: pd.DataFrame = get_production_data()
assert prod_df["dt"].is_monotonic_increasing, "data not sorted by datetime"

To understand when and how long measurements were skipped, we add a new column `dt_diff` that represents the time that passed since the last measurement had been recorded.
There are time gaps in the series of more than three hours.

In [9]:
# Adding 0 padding at the start for length compatibility
prod_df["dt_diff"] = prod_df["dt"].diff().fillna(pd.Timedelta(0))

prod_df.nlargest(10, "dt_diff")

,dt,actual,xgboost,dt_diff
346,2025-08-22 01:07:46.529000+00:00,0.78,0.97,0 days 03:45:52.708000
754,2025-09-03 00:52:00.707000+00:00,1.09,1.30,0 days 03:36:37.407000
1266,2025-09-18 00:55:43.302000+00:00,0.82,1.44,0 days 03:31:25.460000
2881,2025-11-02 11:33:07.251000+00:00,0.92,1.83,0 days 02:25:55.244000
1334,2025-09-20 01:59:47.312000+00:00,0.84,1.52,0 days 02:23:14.056000
2598,2025-10-26 01:18:29.025000+00:00,0.87,0.85,0 days 02:09:12.414000
2726,2025-10-29 13:19:49.186000+00:00,2.07,1.92,0 days 02:02:15.680000
2916,2025-11-03 11:58:38.043000+00:00,0.64,2.17,0 days 01:53:03.436000
20,2025-08-13 12:28:28.995000+00:00,0.74,2.10,0 days 01:37:04.453000
710,2025-09-01 14:19:53.960000+00:00,1.25,1.50,0 days 01:36:51.608000


We derive a new data frame `prod_reg` that will have an equidistant time index at every full hour and exactly 30 minutes after. We keep the original data capture time in column `orig_dt` for reference.
We also add a column `grid_dist` that contains the 'time shift' applied to the actually captured values (and their respective capture times) to place them conformant to the grid

In [10]:
prod_reg = prod_df.copy(deep=True)
prod_reg["orig_dt"] = prod_reg["dt"]
prod_reg = (
    prod_reg
    .set_index("dt")
    .resample("30min").nearest(limit=1)
    .rename(columns={"index": "dt"})
)

prod_reg["grid_dist"] = (prod_reg.index.to_series() - prod_reg["orig_dt"]).abs()

prod_reg.head(8)

,actual,xgboost,dt_diff,orig_dt,grid_dist
dt,,,,,
2025-08-13 00:00:00+00:00,1.67,1.40,0 days 00:00:00,2025-08-13 00:25:31.218000+00:00,0 days 00:25:31.218000
2025-08-13 00:30:00+00:00,1.67,1.40,0 days 00:00:00,2025-08-13 00:25:31.218000+00:00,0 days 00:04:28.782000
2025-08-13 01:00:00+00:00,1.76,1.54,0 days 00:34:59.421000,2025-08-13 01:00:30.639000+00:00,0 days 00:00:30.639000
2025-08-13 01:30:00+00:00,1.71,1.46,0 days 00:35:22.547000,2025-08-13 01:35:53.186000+00:00,0 days 00:05:53.186000
2025-08-13 02:00:00+00:00,1.82,1.52,0 days 00:33:49.761000,2025-08-13 02:09:42.947000+00:00,0 days 00:09:42.947000
2025-08-13 02:30:00+00:00,1.62,1.39,0 days 00:37:07.098000,2025-08-13 02:46:50.045000+00:00,0 days 00:16:50.045000
2025-08-13 03:00:00+00:00,1.62,1.39,0 days 00:37:07.098000,2025-08-13 02:46:50.045000+00:00,0 days 00:13:09.955000
2025-08-13 03:30:00+00:00,1.71,1.39,0 days 00:35:15.439000,2025-08-13 03:22:05.484000+00:00,0 days 00:07:54.516000


We only allowed to 'shift' measured values to the equidistant grid for a time difference of max +/- 30   minutes. For all longer gaps, there are still NaNs:

In [11]:
prod_reg[prod_reg.actual.isna()]["actual"].head(20)

dt
2025-08-13 11:30:00+00:00   NaN
2025-08-13 23:30:00+00:00   NaN
2025-08-18 02:00:00+00:00   NaN
2025-08-19 01:00:00+00:00   NaN
2025-08-19 02:30:00+00:00   NaN
2025-08-19 22:00:00+00:00   NaN
2025-08-20 00:30:00+00:00   NaN
2025-08-20 23:30:00+00:00   NaN
2025-08-21 02:00:00+00:00   NaN
2025-08-21 03:30:00+00:00   NaN
2025-08-21 22:00:00+00:00   NaN
2025-08-21 22:30:00+00:00   NaN
2025-08-21 23:00:00+00:00   NaN
2025-08-21 23:30:00+00:00   NaN
2025-08-22 00:00:00+00:00   NaN
2025-08-22 00:30:00+00:00   NaN
2025-08-22 03:00:00+00:00   NaN
2025-08-22 23:00:00+00:00   NaN
2025-08-23 00:30:00+00:00   NaN
2025-08-24 00:00:00+00:00   NaN
Name: actual, dtype: float64

We forward-fill the `orig_dt` and `dt_diff` columns for reference in longer gaps and add a column `actual_int` that applies linear interpolation for the consumption value based on the time index

In [12]:
prod_reg["orig_dt"] = prod_reg["orig_dt"].ffill()
prod_reg["dt_diff"] = prod_reg["dt_diff"].ffill()
prod_reg["grid_dist"] = (prod_reg.index.to_series() - prod_reg["orig_dt"]).abs() # re-compute after forward filling
prod_reg["actual_int"] = prod_reg["actual"].interpolate("time")

# flag sequences: non-NaN, then >=`min_length` NaNs, then next non-NaN (mark the whole span)
def _compute_long_gap_flags(series: pd.Series, min_length: int = 3) -> pd.Series:
    na = prod_reg["actual"].isna()
    grp = na.ne(na.shift()).cumsum()
    run_len = na.groupby(grp).transform("size")
    mask_nan_run = na & (run_len >= min_length)
    start_mask = mask_nan_run & ~mask_nan_run.shift(fill_value=False)
    end_mask = mask_nan_run & ~mask_nan_run.shift(-1, fill_value=False)
    prev_of_start = start_mask.shift(-1, fill_value=False)
    next_of_end = end_mask.shift(1, fill_value=False)
    return (mask_nan_run | prev_of_start | next_of_end).astype(bool)

prod_reg["long_gap_flag"] = _compute_long_gap_flags(prod_reg["actual"])

prod_reg[prod_reg["long_gap_flag"]][["actual", "actual_int", "grid_dist", "orig_dt"]].head(25)

,actual,actual_int,grid_dist,orig_dt
dt,,,,
2025-08-21 21:30:00+00:00,1.54,1.540000,0 days 00:08:06.179000,2025-08-21 21:21:53.821000+00:00
2025-08-21 22:00:00+00:00,NaN,1.431429,0 days 00:38:06.179000,2025-08-21 21:21:53.821000+00:00
2025-08-21 22:30:00+00:00,NaN,1.322857,0 days 01:08:06.179000,2025-08-21 21:21:53.821000+00:00
2025-08-21 23:00:00+00:00,NaN,1.214286,0 days 01:38:06.179000,2025-08-21 21:21:53.821000+00:00
2025-08-21 23:30:00+00:00,NaN,1.105714,0 days 02:08:06.179000,2025-08-21 21:21:53.821000+00:00
2025-08-22 00:00:00+00:00,NaN,0.997143,0 days 02:38:06.179000,2025-08-21 21:21:53.821000+00:00
2025-08-22 00:30:00+00:00,NaN,0.888571,0 days 03:08:06.179000,2025-08-21 21:21:53.821000+00:00
2025-08-22 01:00:00+00:00,0.78,0.780000,0 days 00:07:46.529000,2025-08-22 01:07:46.529000+00:00
2025-09-02 21:30:00+00:00,1.76,1.760000,0 days 00:14:36.700000,2025-09-02 21:15:23.300000+00:00
